# Optuna + `SEFRBoostClassifier`

Tune [`SEFRBoostClassifier`](https://github.com/LinearBoost/linearboost-classifier) (gradient boosting with SEFR oblique splits) using [Optuna](https://optuna.org/) on sklearn’s **Breast Cancer Wisconsin** dataset (binary).

**Install (if needed):** `pip install linearboost optuna scikit-learn` — or install this repo editable: `pip install -e .` from the repository root.

In [ ]:
import warnings

import numpy as np
import optuna
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from linearboost import SEFRBoostClassifier

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

## 1. Load data and train / test split

`SEFRBoostClassifier` expects **dense numeric** input; we use `StandardScaler` in a pipeline.

In [ ]:
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)
print("Train:", X_train.shape, "Test:", X_test.shape, "Classes:", np.unique(y))

## 2. Quick baseline (default hyperparameters)

`Pipeline(StandardScaler → SEFRBoostClassifier)`.

In [ ]:
baseline = Pipeline([
    ("scale", StandardScaler()),
    ("clf", SEFRBoostClassifier(n_estimators=50, random_state=42)),
])
baseline.fit(X_train, y_train)
y_pred = baseline.predict(X_test)
y_proba = baseline.predict_proba(X_test)[:, 1]
print("Baseline F1 (weighted):", f1_score(y_test, y_pred, average="weighted"))
print("Baseline ROC-AUC:", roc_auc_score(y_test, y_proba))

## 3. Optuna: maximize cross-validated F1

Objective: suggest tree size, learning rate, depth, leaf constraints, and subsample; evaluate with **5-fold stratified CV** on the training set only (fast enough for local runs).

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


def objective(trial: optuna.Trial) -> float:
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 20, 150),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 6),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 5, 40),
        "min_samples_split": trial.suggest_int("min_samples_split", 10, 80),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "random_state": 42,
    }
    pipe = Pipeline([
        ("scale", StandardScaler()),
        ("clf", SEFRBoostClassifier(**params)),
    ])
    scores = []
    for train_idx, val_idx in cv.split(X_train, y_train):
        pipe.fit(X_train[train_idx], y_train[train_idx])
        pred = pipe.predict(X_train[val_idx])
        scores.append(f1_score(y_train[val_idx], pred, average="weighted"))
    return float(np.mean(scores))


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30, show_progress_bar=True)
print("Best trial:", study.best_trial.number, "F1 (CV mean):", study.best_value)
print("Best params:", study.best_params)

## 4. Fit tuned model on full training set and evaluate on held-out test

In [ ]:
best = study.best_params.copy()
best["random_state"] = 42
tuned = Pipeline([
    ("scale", StandardScaler()),
    ("clf", SEFRBoostClassifier(**best)),
])
tuned.fit(X_train, y_train)
y_pred_t = tuned.predict(X_test)
y_proba_t = tuned.predict_proba(X_test)[:, 1]
print("Tuned F1 (weighted):", f1_score(y_test, y_pred_t, average="weighted"))
print("Tuned ROC-AUC:", roc_auc_score(y_test, y_proba_t))